In [ ]:
from ad_afqmc_prototype import config
from pyscf import gto, scf, cc
config.configure_once()
# from ad_afqmc_prototype.afqmc import Afqmc

a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = 'sto6g'

atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,basis=basis,spin=spin*nc,unit=unit,verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mycc = cc.CCSD(mf)
mycc.frozen = 0  # freeze O 1s core
mycc.kernel()
et = mycc.ccsd_t()  # for comparison
print(f"CCSD(T) total energy: {mycc.e_tot + et}")

# afqmc = Afqmc(mycc)
# afqmc.n_walkers = 100  # number of walkers
# afqmc.n_eql_blocks = 10  # number of equilibration blocks
# afqmc.n_blocks = 100  # number of sampling blocks
# afqmc.mixed_precision = False
# mean, err = afqmc.kernel()

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#7920-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Fri Mar 20 21:16:44 2026
PySCF version 2.8.0
PySCF path  /home/yichi/research/software/lno_pyscf
GIT HEAD (branch master) ef75f4190e4de208685670651dc6c467f72b6794

[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = b
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000

SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpq2z8z6xl
max_memory 4000 MB (current use 504 MB)
Set gradient conv threshold to 3.16228e-05
Initial guess from minao.
init E= -0.684162336534414
  HOMO = -0.383125316267418  LUMO = 0.315906645079403
cycle= 1 E= -1.05642988216974  delta_E= -0.372  |g|= 5.55e-16  |ddm|= 0.914
  HOMO = -0.470060798713134  LUMO = 0.414892072584251
cycle= 2 E= -1.05642988216974  delta_E= -2.22e-16  |g|= 1.11e-16  |ddm|= 1.3e-15
  HOMO = -0.470060798713134  LUMO = 0.414892072584251
Extra cycle  E= -1.05642988216974  delta_E= -2.22e-16  |g|= 2.22e-16  |ddm|= 2.72e-16
converged SCF energy = -1.05642988216974

******** <class 'pyscf.cc.ccsd.CCSD'> ********
CC2 = 0
CCSD nocc = 1, nmo = 2
frozen orbitals 0
max_cycle = 50
direct = 0
conv_tol = 1e-07
conv_tol_normt = 1e-05
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 4000 MB (current use 507

In [5]:
from __future__ import annotations
from ad_afqmc_prototype.config import configure_once
configure_once()
import dataclasses
from functools import partial
from pathlib import Path
from typing import Any, Callable, Union

import numpy as np

print = partial(print, flush=True)

from ad_afqmc_prototype.core.system import WalkerKind
from ad_afqmc_prototype.prop.types import QmcParams, QmcParamsBase, QmcParamsFp
from ad_afqmc_prototype.setup import Job
from ad_afqmc_prototype.setup import setup as setup_job
from ad_afqmc_prototype.setup_fp import JobFp
from ad_afqmc_prototype.setup_fp import setup_fp as setup_job_fp
from ad_afqmc_prototype.staging import StagedInputs, _is_cc_like
from ad_afqmc_prototype.staging import dump as dump_staged
from ad_afqmc_prototype.staging import load as load_staged
from ad_afqmc_prototype.staging import stage as stage_inputs
from ad_afqmc_prototype.afqmc import banner_afqmc
from ad_afqmc_prototype.afqmc import Afqmc

class myAfqmc:
    """
    AFQMC driver object.

    Parameters
    ----------
    mf_or_cc : Any
        Mean-field or coupled-cluster object from which to build Hamiltonian and trial wavefunction.
    norb_frozen : int, optional
        Number of orbitals to freeze (from the bottom), by default 0 or cc.frozen
        if mf_or_cc is a Pyscf SCF or CC instance, respectively. For CC instances,
        norb_frozen cannot be set to a value differing fron cc.frozen.
    chol_cut : float, optional
        Cholesky decomposition cutoff, by default 1e-5
    cache : Union[str, Path], optional
        Path to cache file for staged inputs, by default None
    n_eql_blocks : int, optional
        Number of equilibration blocks if params is not provided, by default 20
    n_blocks : int, optional
        Number of production blocks if params is not provided, by default 200
    seed : int | None, optional
        Random seed if params is not provided, by default None
    dt : float | None, optional
        Time step if params is not provided, by default None
    n_walkers : int | None, optional
        Number of walkers if params is not provided, by default None
    n_chunk : int | None, optional
        Number of chunks if params is not provided, by default 1
    """

    def __init__(
        self,
        mf_or_cc: Any,
        *,
        norb_frozen: int | None = None,
        chol_cut: float = 1e-5,
        cache: Union[str, Path] | None = None,
        n_eql_blocks: int | None = None,
        n_blocks: int | None = None,
        seed: int | None = None,
        dt: float | None = None,
        n_walkers: int | None = None,
        n_chunks: int | None = None,
    ):
        self._obj = mf_or_cc
        self._cc: Any = None
        if _is_cc_like(mf_or_cc):
            self._cc = mf_or_cc
            self._scf = mf_or_cc._scf
            self.source_kind = "cc"
        else:
            self._scf = mf_or_cc
            self.source_kind = "mf"

        self.norb_frozen = norb_frozen
        self.chol_cut = float(chol_cut)
        self.cache = Path(cache).expanduser().resolve() if cache is not None else None
        self.overwrite_cache = False
        self.verbose = False

        self.walker_kind: WalkerKind | None = None  # resolved in kernel
        self.mixed_precision = True

        self.params: QmcParamsBase | None = None  # resolved in kernel
        _defaults = QmcParams()
        self.dt = _defaults.dt if dt is None else dt
        self.n_walkers = _defaults.n_walkers if n_walkers is None else n_walkers
        self.n_blocks = _defaults.n_blocks if n_blocks is None else n_blocks
        self.n_eql_blocks = _defaults.n_eql_blocks if n_eql_blocks is None else n_eql_blocks
        self.seed = _defaults.seed if seed is None else seed
        self.n_chunks = _defaults.n_chunks if n_chunks is None else n_chunks

        self._staged: StagedInputs | None = None
        self._job: Job | None = None
        self._cache_key: tuple | None = None

        self.e_tot: Any = None
        self.e_err: Any = None
        self.block_energies: Any = None
        self.block_weights: Any = None

    @property
    def staged(self) -> StagedInputs | None:
        return self._staged

    @property
    def job(self) -> Job | None:
        return self._job

    def _dump_params(self, params: QmcParams) -> None:
        assert isinstance(
            params, QmcParams
        ), f"Expected a QmcParams instance, but got {type(params)}"
        fields = dataclasses.fields(params)
        width = len(max(fields, key=lambda f: len(f.name)).name)
        print(" QmcParams:")
        for field in fields:
            print(f"  {field.name:<{width}} = {getattr(params, field.name)}")
        print("")

    def dump_flags(self, job) -> None:
        assert isinstance(job, Job), f"Expected a Job instance, but got {type(job)}"
        self._dump_flags_helper(job)

    def _dump_flags_helper(self, job) -> None:
        meta = job.staged.meta
        src = meta["source_kind"]
        chol_cut = meta["chol_cut"]
        sys = job.sys
        nchol = job.staged.ham.chol.shape[0]
        params = job.params
        trial = job.staged.trial
        print("******** AFQMC ********")
        print(f" norb            = {sys.norb}")
        print(f" nelec_up        = {sys.nelec[0]}")
        print(f" nelec_dn        = {sys.nelec[1]}")
        print(f" nchol           = {nchol}")
        print(f" source_kind     = {src}")
        print(f" trial_kind      = {trial.kind}")
        print(f" chol_cut        = {chol_cut:g}")
        print(f" cache           = {str(self.cache) if self.cache else None}")
        print(f" walker_kind     = {sys.walker_kind}")
        print(f" mixed_precision = {self.mixed_precision}\n")
        self._dump_params(params)

    def _key(self) -> tuple:
        """Key for determining whether staged/job caches are still valid."""
        cache_mtime = None
        if self.cache is not None and self.cache.exists():
            cache_mtime = self.cache.stat().st_mtime
        return (
            self.source_kind,
            self.norb_frozen,
            float(self.chol_cut),
            str(self.cache) if self.cache is not None else None,
            bool(self.overwrite_cache),
            cache_mtime,
        )

    def stage(self, *, force: bool = False) -> StagedInputs:
        """
        Compute or load HamInput/TrialInput.
        If cache is set and exists, loads unless overwrite_cache=True.
        """
        key = self._key()
        if self._staged is not None and self._cache_key == key and not force:
            return self._staged

        staged = stage_inputs(
            self._obj,
            norb_frozen=self.norb_frozen if self.norb_frozen is not None else None,
            chol_cut=self.chol_cut,
            cache=self.cache,
            overwrite=self.overwrite_cache if self.cache is not None else False,
            verbose=self.verbose,
        )
        self._staged = staged
        self._cache_key = key
        self._job = None
        return staged

    def save_staged(self, path: Union[str, Path]) -> None:
        """Write current staged inputs to a single file cache."""
        staged = self.stage()
        dump_staged(staged, path)

    # def load_staged(self, path: Union[str, Path]): -> StagedInputs:
    #    """Load staged inputs from a cache file and attach them to this object."""
    #    staged = load_staged(path)
    #    self._staged = staged
    #    self._cache_key = None
    #    self._job = None
    #    return staged

    def _make_params(self) -> QmcParams:
        """
        Create QmcParams if user didn't provide one.
        """
        if self.params is not None and isinstance(self.params, QmcParams):
            params = self.params
        elif self.params is not None and not isinstance(self.params, QmcParams):
            raise TypeError(
                f"Expected type QmcParams for self.params, but received '{type(self.params)}'"
            )
        else:
            kwargs: dict[str, Any] = {}
            for field in dataclasses.fields(QmcParams):
                if hasattr(self, field.name):
                    val = getattr(self, field.name)
                    if val is not None:
                        kwargs[field.name] = val

            params = QmcParams(**kwargs)

        return params

    def build_job(
        self,
        *,
        force: bool = False,
        trial_data: Any = None,
        trial_ops: Any = None,
        meas_ops: Any = None,
        prop_ops: Any = None,
        block_fn: Callable[..., Any] | None = None,
        prop_kwargs: dict[str, Any] | None = None,
    ) -> Job:
        """
        Assemble a runnable Job from current settings and staged inputs.
        """
        if self._job is not None and not force:
            return self._job

        staged = self.stage()
        qmc_params = self._make_params()
        self.params = qmc_params

        job = setup_job(
            staged,
            walker_kind=self.walker_kind,
            mixed_precision=self.mixed_precision,
            params=qmc_params,
            trial_data=trial_data,
            trial_ops=trial_ops,
            meas_ops=meas_ops,
            prop_ops=prop_ops,
            block_fn=block_fn,
            prop_kwargs=prop_kwargs,
        )
        self._job = job
        return job

    def kernel(self, **driver_kwargs: Any) -> tuple[float, float]:
        """
        Runs AFQMC, returns (e_tot, e_err), and stores samples.
        """
        print(banner_afqmc())
        job = self.build_job()
        self.dump_flags(job)

        out = job.kernel(**driver_kwargs)

        if isinstance(out, tuple) and len(out) >= 2:
            e_tot = float(out[0])
            e_err = float(out[1])
            block_e = out[2] if len(out) > 2 else None
            block_w = out[3] if len(out) > 3 else None
        else:
            raise TypeError("Unexpected return from Job.kernel(), expected tuple output.")

        self.e_tot = e_tot
        self.e_err = e_err
        self.block_energies = block_e
        self.block_weights = block_w
        return e_tot, e_err

    run = kernel

    @classmethod
    def from_staged(
        cls,
        path: Union[str, Path],
        *,
        n_eql_blocks: int | None = None,
        n_blocks: int | None = None,
        seed: int | None = None,
        dt: float | None = None,
        n_walkers: int | None = None,
        n_chunks: int = 1,
    ) -> Afqmc:
        """
        Returns a new AFQMC object from a previously staged calculations
        (using save_staged method). The number of frozen orbitals, norb_frozen,
        and the choliesky decomposition threshold, chol_cut, cannot be changed.
        Parameters
        ----------
        path: str, pathlib.Path
        The other parameters are identical to the ones in the AFQMC class.
        """
        staged = load_staged(path)
        meta = staged.meta

        mf_or_cc = None

        # Cannot be changed as the input has been staged
        norb_frozen = meta["norb_frozen"]
        chol_cut = meta["chol_cut"]

        af = Afqmc(
            mf_or_cc,
            norb_frozen=norb_frozen,
            chol_cut=chol_cut,
            n_eql_blocks=n_eql_blocks,
            n_blocks=n_blocks,
            seed=seed,
            dt=dt,
            n_walkers=n_walkers,
            n_chunks=n_chunks,
        )

        af._staged = staged
        af.source_kind = meta["source_kind"]
        af._cache_key = af._key()

        return af

In [7]:
myqmc = myAfqmc(mycc)
myqmc.dt = 0.005
myqmc.seed = 17
myqmc.n_walkers = 100  # number of walkers
myqmc.n_eql_blocks = 10  # number of equilibration blocks
myqmc.n_blocks = 100  # number of sampling blocks
myqmc.mixed_precision = False

job = myqmc.build_job()
out = myqmc.job.kernel()


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/10]   -1.0960712811  1.000000e+02           0       0.0
[eql    2/10]   -1.0960712825  1.000225e+02           0       3.2
[eql    4/10]   -1.0960712824  9.996510e+01           0       6.1
[eql    6/10]   -1.0960712825  1.000347e+02           0       6.1
[eql    8/10]   -1.0960712820  1.000207e+02           0       6.2
[eql   10/10]   -1.0960712819  9.998372e+01           0       6.2

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   10/100]   -1.0960712826   6.264e-11     -1.0960712826  9.999717e+01           0      0.942       9.4
[blk   20/100]   -1.0960712822   1.228e-10     -1.0960712818  9.999783e+01           0      0.023       9.6
[blk   30/100]   -1.0960712822   1.059e-10     -1.0960712822  1.000013e+02           0      0.023       9.9
[blk   40/100]   -1.0960712822   9.240e-11     -1.0960712824  1.000102e+02   

In [ ]:
job.

<function ad_afqmc_prototype.prop.blocks.block(state: 'PropState', *, sys: 'System', params: 'QmcParams', ham_data: 'Any', trial_data: 'Any', trial_ops: 'TrialOps', meas_ops: 'MeasOps', meas_ctx: 'Any', prop_ops: 'PropOps', prop_ctx: 'Any', sr_fn: 'Callable' = <function stochastic_reconfiguration at 0x7f41dd0188b0>, observable_names: 'tuple[str, ...]' = ()) -> 'tuple[PropState, BlockObs]'>